In [11]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

In [12]:
# Load CSV
try:
    df = pd.read_csv("../../Datasets/csv/UAVUnderNormalCyberattacks.csv", low_memory=False)
except FileNotFoundError:
    print("Error: The file '../Datasets/csv/UAVUnderNormalCyberattacks.csv' was not found.")
    print("Please make sure the path is correct.")
    exit()

In [13]:
RANDOM_STATE = 1082
TEST_SIZE = 0.8

In [14]:
# Data Preprocessing

# Drop columns
drop_cols = ['timestamp_c', 'class']

# Print unique labels before encoding
print("Unique labels before encoding:", df['class'].unique())

# Remove rows where 'class' is NaN or is the string 'class'
df = df[df['class'].notna()]
df = df[df['class'] != 'class']

# Print unique labels again after cleaning
print("Unique labels after cleaning:", df['class'].unique())

X = df.drop(columns=drop_cols).apply(pd.to_numeric, errors='coerce')

# Encode labels cause of string values
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(df['class'])

print("Unique classes after encoding:", np.unique(y))
num_classes = len(np.unique(y))

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

Unique labels before encoding: ['benign' nan 'class' 'DoS attack' 'Replay']
Unique labels after cleaning: ['benign' 'DoS attack' 'Replay']
Unique classes after encoding: [0 1 2]


In [15]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

# Build and train XGBoost classifier
model = xgb.XGBClassifier(
    n_estimators=100,
    eval_metric='mlogloss',
    random_state=RANDOM_STATE,
    objective='multi:softprob',
    num_class=num_classes
)

model.fit(X_train, y_train)

# Predict and evaluate
y_pred = model.predict(X_test)
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_ if 'le' in locals() else None))

Confusion Matrix:
[[8254  659  424]
 [ 689 8464  452]
 [  87  143 7310]]

Classification Report:
              precision    recall  f1-score   support

  DoS attack       0.91      0.88      0.90      9337
      Replay       0.91      0.88      0.90      9605
      benign       0.89      0.97      0.93      7540

    accuracy                           0.91     26482
   macro avg       0.91      0.91      0.91     26482
weighted avg       0.91      0.91      0.91     26482

